# Python autograd — `tensor.autograd` from CPython

*Tape-based reverse-mode automatic differentiation over named-axis tensors, reached for from Python instead of C++. Phase 6 M4 per [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md) + the [Phase 6 impl-plan](../../docs/impl-plans/2026-05-12_phase-6-python-sdk.md).*

This notebook is the Python-side companion to [`tutorials/05_autograd-from-scratch.ipynb`](../../tutorials/05_autograd-from-scratch.ipynb) and [`tutorials/07_mlp-on-toy.ipynb`](../../tutorials/07_mlp-on-toy.ipynb). The C++ Domain's autograd subsystem is consumed directly from Python — same Tape, same backward closures, same `gradient_check` discipline.

## Prerequisites

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — `Axis`, `DynamicShape`, `DynamicTensor`, NumPy interop.

## What this notebook covers

1. **`DynamicVariable`** — wrap a `DynamicTensor` and tell the tape to track it.
2. **Forward + backward** — `loss = Σ x_i²` → `dL/dx_i = 2 x_i`, the canonical autograd sanity check.
3. **Activations** — `exp` / `log` / `relu` / `neg` with closed-form gradients.
4. **`dot` — autograd-aware contract** — `y = W·x` with `∂L/∂x` computed by the same kernel.
5. **Linear regression on `y = 2x + 1`** — full training loop end-to-end; exit criterion `|W - 2| < 1e-3` and `|b - 1| < 1e-3`.
6. **Where this fits** — milestone map, sibling notebooks, cross-references.

In [ ]:
# Colab / Binder setup — install the tensor SDK on first run.
try:
    import tensor  # noqa: F401
except ImportError:
    import subprocess as _sp
    _sp.run(
        ["pip", "install", "--quiet",
         "git+https://github.com/uyuutosa/tensor.git"],
        check=True,
    )
    import tensor  # noqa: F401

## 1. Setup + `DynamicVariable`

A `DynamicVariable` wraps a `DynamicTensor`. When `requires_grad=True`, the tape allocates a gradient accumulator alongside; every op the variable participates in registers a backward closure.

In [ ]:
import numpy as np

import tensor
import tensor.autograd as ag

x_np = np.array([1.0, 2.0, 3.0, 4.0])
x = ag.DynamicVariable(
    tensor.from_numpy(x_np, ["i"]),
    requires_grad=True,
)
print(f"value =\n{x.value}")
print(f"requires_grad = {x.requires_grad}")
print(f"grad (zero-initialised) = {x.grad.numpy()}")

## 2. Forward + backward — the canonical sanity check

Define `L(x) = Σ_i x_i²`. The closed form is `dL/dx_i = 2 x_i`. Build the graph forward, walk it backward, compare.

In [ ]:
# Forward: build the graph through ops the tape understands.
loss = ag.sum_all(x * x)
print(f"loss = {loss.value:.4f}  (expected: 1 + 4 + 9 + 16 = 30)")

# Backward: walk the tape.
ag.backward(loss)
print(f"x.grad = {x.grad.numpy()}")
print(f"expected (2x) = {2 * x_np}")

np.testing.assert_allclose(x.grad.numpy(), 2 * x_np, atol=1e-12)
print("\nForward + backward agree analytically to 1e-12.")

## 3. Activations — closed-form gradients

Four activation functions ship at M4: `exp`, `log`, `relu`, `neg`. Each has a backward closure that consults the forward-pass value (so the derivative does not need to be re-derived at backward time).

In [ ]:
# d/dx exp(x) = exp(x)
x_np = np.array([0.0, 1.0, -0.5])
x = ag.DynamicVariable(tensor.from_numpy(x_np, ["i"]), requires_grad=True)
ag.backward(ag.sum_all(ag.exp(x)))
print(f"d/dx exp(x):\n  got      {x.grad.numpy()}\n  expected {np.exp(x_np)}")
np.testing.assert_allclose(x.grad.numpy(), np.exp(x_np), atol=1e-12)

In [ ]:
# ReLU: derivative is the indicator function 1{x > 0}.
x_np = np.array([-2.0, -1.0, 0.5, 2.0])
x = ag.DynamicVariable(tensor.from_numpy(x_np, ["i"]), requires_grad=True)
ag.backward(ag.sum_all(ag.relu(x)))
expected = (x_np > 0.0).astype(np.float64)
print(f"d/dx relu(x):\n  got      {x.grad.numpy()}\n  expected {expected}")

## 4. `dot` — autograd-aware named-axis contraction

`ag.dot(W, x)` is the autograd counterpart of `tensor.contract`. Forward: shared-axis Einstein-sum. Backward: the same contract kernel re-purposed for the gradient. No new C++ code needed beyond the shared-axis math.

Below: `y = W · x` (matrix-vector), then `L = Σ_i y_i`. The closed-form `dL/dx_j = Σ_i W_{ij}` is a column-sum of W.

In [ ]:
W_np = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
x_np = np.array([10.0, 20.0, 30.0])
W = ag.DynamicVariable(tensor.from_numpy(W_np, ["i", "j"]))
x = ag.DynamicVariable(
    tensor.from_numpy(x_np, ["j"]),
    requires_grad=True,
)
y = ag.dot(W, x)
loss = ag.sum_all(y)
ag.backward(loss)

expected = W_np.sum(axis=0)  # column sums
print(f"x.grad   = {x.grad.numpy()}")
print(f"expected = {expected}")
np.testing.assert_allclose(x.grad.numpy(), expected, atol=1e-12)

## 5. Linear regression — the M4 exit criterion

The headline cell. Mirrors [`tutorials/07_mlp-on-toy.ipynb`](../../tutorials/07_mlp-on-toy.ipynb) §4 in Python — fit `y = 2x + 1` from a noisy sample. Exit: `|W - 2| < 1e-3` and `|b - 1| < 1e-3` after 500 epochs at lr = 0.1.

`W` and `b` are rank-0 (scalar) DynamicVariables; Einstein-style broadcast against the rank-1 `x_var` produces rank-1 predictions automatically. Every iteration: forward → loss → backward → `sgd_update` → re-wrap as a fresh `DynamicVariable` (the wrap step gives us a fresh tape entry; this is the M4 ergonomic — M5/M6 may add an in-place update helper).

In [ ]:
# Toy data: y = 2x + 1 + tiny noise
rng = np.random.default_rng(seed=1729)
n = 64
x_data = np.linspace(-1.0, 1.0, n)
noise = rng.normal(0.0, 0.001, size=n)
y_data = 2.0 * x_data + 1.0 + noise

x_var = ag.DynamicVariable(tensor.from_numpy(x_data, ["n"]))
y_var = ag.DynamicVariable(tensor.from_numpy(y_data, ["n"]))

W = ag.DynamicVariable(tensor.from_numpy(np.array(0.0), []), requires_grad=True)
b = ag.DynamicVariable(tensor.from_numpy(np.array(0.0), []), requires_grad=True)
inv = ag.DynamicVariable(tensor.from_numpy(np.array(1.0 / n), []))

lr = 0.1
history = []
for epoch in range(500):
    W.zero_grad()
    b.zero_grad()
    pred = W * x_var + b
    diff = pred - y_var
    loss = ag.sum_all(diff * diff * inv)
    ag.backward(loss)
    W = ag.DynamicVariable(ag.sgd_update(W, lr), requires_grad=True)
    b = ag.DynamicVariable(ag.sgd_update(b, lr), requires_grad=True)
    history.append(loss.value)

final_W = float(W.value.numpy())
final_b = float(b.value.numpy())
print(f"final W = {final_W:.6f}  (target 2.0)")
print(f"final b = {final_b:.6f}  (target 1.0)")
print(f"final loss = {history[-1]:.2e}  (initial {history[0]:.2e})")

assert abs(final_W - 2.0) < 1e-3
assert abs(final_b - 1.0) < 1e-3
print("\nExit criterion met: |W - 2| < 1e-3 and |b - 1| < 1e-3.")

## 6. Where this fits

**Phase 6 milestone map**:

| Milestone | Surface | Status (2026-05-13) |
| --------- | ------- | -------------------- |
| P6.M1 | scaffold + smoke binding | ✅ shipped |
| P6.M2 | `Axis` + `DynamicShape` + `DynamicTensor` + arithmetic | ✅ shipped |
| P6.M3 | `contract` + NumPy interop | ✅ shipped |
| **P6.M4** | **autograd (this notebook)** | **✅ shipped** |
| P6.M5 | `tex.parse` + `Evaluator` (the `_tex` UDL equivalent) | next |
| P6.M6 | runtime backend selection + `0.2.0` release | exit |

**Sibling notebooks**:

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — Python SDK M1..M3 tour (the prerequisite).
- [`tutorials/05_autograd-from-scratch.ipynb`](../../tutorials/05_autograd-from-scratch.ipynb) — the C++ autograd primitive-by-primitive narration this Python surface mirrors.
- [`tutorials/07_mlp-on-toy.ipynb`](../../tutorials/07_mlp-on-toy.ipynb) — the C++ training-loop §4 that §5 above reproduces in Python.

**Cross-references**:

- [ADR-0007](../../docs/arc42/09-decisions/0007-adopt-autograd-as-first-class-subsystem.md) — autograd as a first-class subsystem.
- [`docs/detailed-design/tensor-autograd.md`](../../docs/detailed-design/tensor-autograd.md) — the autograd implementation HOW (tape, registered backwards, broadcast-aware unbroadcast).
- [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md) — Phase 6 / Python SDK entry decisions.